In [7]:
import os
import sys
sys.path.append('../')
from Utils.download import read_raws
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve
import pandas as pd
import numpy as np
from scipy.signal import welch
from Utils.train import patient_stratify_split
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

In [3]:
path_demographic="../d/data_patients.csv"
channels=['Fp1', 'Fp2','F3','F4','C3','C4','P3','P4','O1','O2','F7','F8','T3','T4','T5','T6','Fz','Cz','Pz']
channels_dif=['Fp1-F3', 'F3-C3', 'C3-P3', 'P3-O1', 'Fp2-F4', 'F4-C4', 'C4-P4', 'P4-O2', 'F7-T3', 'T3-T5', 'T5-O1', 'F8-T4', 'T4-T6', 'T6-O2', 'Fz-Cz', 'Cz-Pz']

### Data base demographic
df_demog=pd.read_csv(path_demographic, sep=";" )
df_demog["numeric_outcome"]=df_demog["Outcome"].map({"Good": 1, "Poor": 0})
df_demog.reset_index(drop=True, inplace=True)
df_demog.head()
scaler=StandardScaler()
general_path="../raw_tif/"
files=[i+"/" for i in os.listdir(general_path)]

In [20]:
general_path="../raw_tif/"
files=[i+"/" for i in os.listdir(general_path)]
df=pd.DataFrame()
for f in files[5:10]:
    raw_list, patients= read_raws(path_files=general_path+f, time_taken=180, del_files=False)
    data=[]
    aux_data=pd.DataFrame()
    for k, raw in enumerate(raw_list):
        df_aux=pd.DataFrame(raw.copy().get_data().T, columns=raw.copy().ch_names)
        df_aux=df_aux[channels_dif]
        mean=pd.DataFrame(df_aux.mean().values.reshape(1, -1), columns=df_aux.mean().index+"_mean")
        var=pd.DataFrame(df_aux.var().values.reshape(1, -1), columns=df_aux.var().index+"_var")
        car=pd.DataFrame()
        car[df_demog.columns]=df_demog[df_demog["Id_Patient"]==int(patients[k])].values
        car[mean.columns]=mean.values
        car[var.columns]=var.values
        aux_data=pd.concat([aux_data,car])
    aux_data.set_index("Id_Patient",inplace=True)
    del data , raw_list

    df=pd.concat([df,aux_data])
df["indices"]=range(len(df))

Opening raw data file ../raw_tif/t180_h34/patient_0284.fif...
    Range : 0 ... 23039 =      0.000 ...   179.992 secs
Ready.
Reading 0 ... 23039  =      0.000 ...   179.992 secs...
Opening raw data file ../raw_tif/t180_h34/patient_0286.fif...
    Range : 0 ... 23039 =      0.000 ...   179.992 secs
Ready.
Reading 0 ... 23039  =      0.000 ...   179.992 secs...
Opening raw data file ../raw_tif/t180_h34/patient_0299.fif...
    Range : 0 ... 23039 =      0.000 ...   179.992 secs
Ready.
Reading 0 ... 23039  =      0.000 ...   179.992 secs...
Opening raw data file ../raw_tif/t180_h34/patient_0303.fif...
    Range : 0 ... 23039 =      0.000 ...   179.992 secs
Ready.
Reading 0 ... 23039  =      0.000 ...   179.992 secs...
Opening raw data file ../raw_tif/t180_h34/patient_0306.fif...
    Range : 0 ... 23039 =      0.000 ...   179.992 secs
Ready.
Reading 0 ... 23039  =      0.000 ...   179.992 secs...
Opening raw data file ../raw_tif/t180_h34/patient_0312.fif...
    Range : 0 ... 23039 =      0.

In [21]:
patients_train, patients_val= patient_stratify_split(df, train_size=0.75)
X_train=df.loc[patients_train].drop(columns=['Hospital', 'Age', 'Sex', 'ROSC', 'OHCA', 'Shockable Rhythm', 'TTM',
       'Outcome', 'CPC', 'numeric_outcome', 'indices'])
y_train=pd.to_numeric(df.loc[patients_train]['numeric_outcome'])

X_val=df.loc[patients_val].drop(columns=['Hospital', 'Age', 'Sex', 'ROSC', 'OHCA', 'Shockable Rhythm', 'TTM',
       'Outcome', 'CPC', 'numeric_outcome', 'indices'])
y_val=pd.to_numeric(df.loc[patients_val]['numeric_outcome'])

In [23]:
model=RandomForestClassifier()
model.fit(X_train, y_train)
y_pred=model.predict(X_val)
print(classification_report(y_val,y_pred))

              precision    recall  f1-score   support

           0       0.65      0.81      0.73       320
           1       0.56      0.36      0.44       214

    accuracy                           0.63       534
   macro avg       0.61      0.59      0.58       534
weighted avg       0.62      0.63      0.61       534



In [24]:
y_score=model.predict_proba(X_val)
fpr, tpr, thresholds = roc_curve(y_val, y_score[:,1])
roc_auc_score(y_val, y_score[:,1])

0.6054468457943925

In [25]:
confusion_matrix(y_val, y_pred, normalize="true")

array([[0.8125    , 0.1875    ],
       [0.64018692, 0.35981308]])